In [15]:
import pandas as pd
import numpy as np

In [16]:
df=pd.read_csv("01 covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [17]:
print(df.cough.value_counts())
print()
print()
print(df.city.value_counts())


cough
Mild      62
Strong    38
Name: count, dtype: int64


city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64


In [18]:
'''
age-numeric column
gender-nominal categorical column(one hot encoder)
fever-numeric column.has missing values.will aplly simple imputer
cough-nominal categorical column(one hot encoder)
city-nominal categorical column(one hot encoder)
has_covid-nominal categorical column(label encoder)

here we will work on nominal categorical columns only.
'''

'\nage-numeric column \ngender-nominal categorical column(one hot encoder)\nfever-numeric column.has missing values.will aplly simple imputer\ncough-nominal categorical column(one hot encoder)\ncity-nominal categorical column(one hot encoder)\nhas_covid-nominal categorical column(label encoder)\n\nhere we will work on nominal categorical columns only.\n'

In [19]:
df.isnull().sum()


,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [20]:
x=df.iloc[:,0:5]
y=df.drop(x,axis='columns')

print(x.head())
print()
print()
print()

print(y.head())


   age  gender  fever cough     city
0   60    Male  103.0  Mild  Kolkata
1   27    Male  100.0  Mild    Delhi
2   42    Male  101.0  Mild    Delhi
3   31  Female   98.0  Mild  Kolkata
4   65  Female  101.0  Mild   Mumbai



  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [52]:
# train-test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
print(x_train.head())
print()
print()
print(y_train.head())


    age  gender  fever   cough       city
84   69  Female   98.0  Strong     Mumbai
19   42  Female    NaN  Strong  Bangalore
97   20  Female  101.0    Mild  Bangalore
25   23    Male    NaN    Mild     Mumbai
48   66    Male   99.0  Strong  Bangalore


   has_covid
84        No
19       Yes
97        No
25        No
48        No


In [22]:
## 1.Without column transform

##performing SimpleImpute on fever column for filling missing value

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder



In [23]:
si=SimpleImputer()
le=LabelEncoder()

In [24]:
## applying label encoding on class column
y_train.has_covid=le.fit_transform(y_train.has_covid)
y_test.has_covid=le.fit_transform(y_test['has_covid'])


In [27]:
x_train['fever']=si.fit_transform(x_train[['fever']])
x_test['fever']=si.transform(x_test[['fever']])

In [35]:
from sklearn.preprocessing import OrdinalEncoder


In [36]:
oe=OrdinalEncoder()

x_train['cough']=oe.fit_transform(x_train[['cough']])
x_test['cough']=oe.transform(x_test[['cough']])

In [37]:
from sklearn.preprocessing import OneHotEncoder

In [38]:
ohe=OneHotEncoder(drop='first',sparse_output=False)

In [39]:
x_train_gender_city=ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city=ohe.transform(x_test[['gender','city']])

In [40]:
print(x_train_gender_city.shape)
print(x_test_gender_city.shape)

(80, 4)
(20, 4)


In [41]:
x_train_gender_city[1]

array([1., 1., 0., 0.])

In [42]:
## Extracting values of age

x_train_age=x_train['age'].values
x_test_age=x_test['age'].values

In [43]:
## Column transformer
from sklearn.compose import ColumnTransformer

In [46]:
## Column transformer() Takes Two Arguments'

##1.transformer and
##2. remainder

In [48]:
transformer=[('trans1',SimpleImputer(),['fever']),
('trans2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
('trans3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])]

In [49]:
ct=ColumnTransformer(transformers=transformer,remainder='passthrough').set_output(transform='pandas')

In [53]:
x_train_trans=ct.fit_transform(x_train)

In [55]:
x_train_trans.sample(5)

,trans1__fever,trans2__cough,trans3__gender_Male,trans3__city_Delhi,trans3__city_Kolkata,trans3__city_Mumbai,remainder__age
11,98.0,0.0,0.0,0.0,0.0,1.0,65
70,101.0,1.0,0.0,1.0,0.0,0.0,68
24,100.0,1.0,0.0,0.0,1.0,0.0,13
47,104.0,0.0,0.0,0.0,0.0,0.0,18
84,98.0,1.0,0.0,0.0,0.0,1.0,69
